# Deep Hedging — Benchmarks : d'où viennent les cibles

Le couvreur neuronal se compare à deux références, calculées ici dans **exactement le même réglage** (S0=K=100, mu=0.10, r=0.02, sigma=0.20, T=1, coût=1%, n=63, alpha=0.95) :

1. **Delta-hedging pur** : rééquilibrer vers le delta de Black-Scholes à chaque date. C'est le benchmark classique, celui de la théorie sans friction.
2. **Bande de non-transaction** : ne rééquilibrer que si l'écart au delta dépasse un seuil `band`, pour économiser les coûts. C'est une heuristique connue sous frictions (Leland 1985, Whalley-Wilmott 1997). On optimise la largeur.

On mesure les deux par la **CVaR 95% de la perte**. Résultats : delta pur $\approx 6.40$, meilleure bande $\approx 5.70$. Ce sont les cibles que le réseau doit battre.

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Simulation, Black-Scholes, et couverture avec bande

Le delta-hedging pur est le cas particulier `band = 0` (on rééquilibre toujours).

In [ ]:
def simulate_gbm(S0, mu, sigma, T, n, m):
    dt = T/n
    Z = rng.standard_normal((m, n))
    inc = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    return S0*np.exp(np.concatenate([np.zeros((m,1)), np.cumsum(inc, axis=1)], axis=1))

def bs_price(S, K, tau, r, s):
    S = np.asarray(S, float)
    d1 = (np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)); d2 = d1 - s*np.sqrt(tau)
    return S*norm.cdf(d1) - K*np.exp(-r*tau)*norm.cdf(d2)

def bs_delta(S, K, tau, r, s):
    S = np.asarray(S, float)
    return norm.cdf((np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)))

def band_hedge_pnl(S, K, T, r, s, cost, band):
    """P&L d'un vendeur de call qui rééquilibre vers le delta SEULEMENT si
    |delta_cible - position| > band. band=0 -> delta-hedging pur."""
    m, n1 = S.shape; n = n1-1; dt = T/n; times = np.linspace(0, T, n1)
    cash = bs_price(S[:,0], K, T, r, s).copy(); sh = np.zeros(m)
    for k in range(n):
        tau = T - times[k]
        target = bs_delta(S[:,k], K, tau, r, s)
        trade = np.where(np.abs(target - sh) > band, target - sh, 0.0)   # bande
        cash -= trade*S[:,k]; cash -= cost*np.abs(trade)*S[:,k]
        sh = sh + trade; cash *= np.exp(r*dt)
    return cash + sh*S[:,-1] - np.maximum(S[:,-1]-K, 0.0)

def cvar(pnl, alpha=0.95):
    loss = -pnl
    return loss[loss >= np.quantile(loss, alpha)].mean()

## Les deux cibles, et le balayage de la largeur de bande

In [ ]:
S0, K, mu, r, s, T = 100., 100., 0.10, 0.02, 0.20, 1.0
cost, n, m = 0.01, 63, 120_000
S = simulate_gbm(S0, mu, s, T, n, m)

bands = np.linspace(0, 0.30, 16)
cs = np.array([cvar(band_hedge_pnl(S, K, T, r, s, cost, b)) for b in bands])
i = int(np.argmin(cs))

print(f"CIBLE 1 — delta pur (band=0) : CVaR = {cs[0]:.3f}")
print(f"CIBLE 2 — meilleure bande = {bands[i]:.3f} : CVaR = {cs[i]:.3f}")

## La forme en U

Le delta pur (bande nulle) paie trop de coûts. Élargir la bande réduit les coûts mais augmente l'erreur de réplication : la CVaR passe par un minimum. C'est ce minimum, `~5.70`, qui est la vraie cible du réseau.

In [ ]:
plt.figure(figsize=(7, 4.6))
plt.plot(bands, cs, "o-", color="navy")
plt.scatter([bands[0]], [cs[0]], s=120, color="crimson", zorder=5, label=f"delta pur = {cs[0]:.2f}")
plt.scatter([bands[i]], [cs[i]], s=140, facecolors="none", edgecolors="green",
            linewidths=2, zorder=5, label=f"bande optimale = {cs[i]:.2f}")
plt.xlabel("largeur de bande (unités de delta)"); plt.ylabel("CVaR 95% de la perte")
plt.title(f"Benchmarks de couverture (coût={cost:.0%}, n={n})")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()